## 犯罪の種類別に計数する

#### 注：このデータ処理では，シカゴ市警のデータに記載されているCommunity Areaをもとに集計する。
#### 　　GISデータとの照合は行っていない。

In [1]:
import pandas as pd

In [975]:
year = 2025
df1 = pd.read_csv(rf"D:\Jupyterlab Raw Data\シカゴ市警察データ\Crimes\Classified\classified_crime{year}.csv")

In [976]:
df1.head()

,ID,case_number,date,block,IUCR,primary_type,description,location_description,crime_group1_prime,crime_group3_iucr,...,Ward,community_area,fbi_code,year,Latitude,Longitude,Location,PRIMARY DESCRIPTION,SECONDARY DESCRIPTION,INDEX CODE
0,14075483,JK105557,12/31/2025 11:58:00 PM,050XX S PAULINA ST,0560,assault,SIMPLE,RESIDENCE,violent1,violent3,...,20.0,61.0,08A,2025,41.802549,-87.667246,POINT (-87.667246428 41.802549018),ASSAULT,SIMPLE,N
1,14070833,JK100050,12/31/2025 11:55:00 PM,053XX W WASHINGTON BLVD,0930,motor_vehicle_theft,THEFT / RECOVERY - AUTOMOBILE,APARTMENT,property1,property3,...,37.0,25.0,07,2025,41.882329,-87.758411,POINT (-87.758411303 41.882328854),MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,I
2,14070845,JK100006,12/31/2025 11:54:00 PM,013XX W LAKE ST,0454,battery,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",RESTAURANT,violent1,violent3,...,27.0,28.0,08B,2025,41.885427,-87.661759,POINT (-87.661759042 41.885426714),BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",N
3,14070745,JK100011,12/31/2025 11:54:00 PM,100XX W OHARE ST,2890,public_peace_violation,OTHER VIOLATION,AIRCRAFT,public_order1,minor_offence3,...,41.0,76.0,24,2025,41.976290,-87.905227,POINT (-87.905227221 41.976290414),PUBLIC PEACE VIOLATION,OTHER VIOLATION,N
4,14070799,JK100014,12/31/2025 11:54:00 PM,100XX W OHARE ST,2890,public_peace_violation,OTHER VIOLATION,AIRCRAFT,public_order1,minor_offence3,...,41.0,76.0,24,2025,41.976290,-87.905227,POINT (-87.905227221 41.976290414),PUBLIC PEACE VIOLATION,OTHER VIOLATION,N


#### 年を表す列を先頭に移動させる。

In [977]:
col = df1.pop("year")
df1.insert(0, "year", col)

In [978]:
# df1["year"].unique()

In [979]:
# df1["year"].dtype

In [980]:
# df1.head()

In [981]:
# df1.nunique()

In [982]:
# df1.columns

In [983]:
df1["primary_type"].unique()

array(['assault', 'motor_vehicle_theft', 'battery',
       'public_peace_violation', 'weapons_violation',
       'deceptive_practice', 'theft', 'robbery', 'criminal_trespass',
       'burglary', 'criminal_damage', 'other_offense', 'narcotics',
       'liquor_law_violation', 'offense_involving_children',
       'sex_offense', 'criminal_sexual_assault', 'stalking', 'kidnapping',
       'interference_with_public_officer', 'arson', 'intimidation',
       'homicide', 'concealed_carry_license_violation', 'prostitution',
       'obscenity', 'human_trafficking', 'public_indecency', 'gambling',
       'other_narcotic_violation'], dtype=object)

In [984]:
# df1[df1["Latitude"].isna()]["primary_type"].nunique()
# df1["Latitude"].isna().sum()

In [985]:
df1['community_area'].isna().sum()

np.int64(9)

In [986]:
# df1['Latitude'].isna().sum()

In [987]:
#community_areaの欠損値を削除する
df2 = df1.dropna(subset=["community_area"])

In [988]:
df2["community_area"].nunique()

77

In [989]:
# df2[df2["community_area"] == 0]

In [990]:
df3 = df2[df2["community_area"] != 0]

In [991]:
df3["community_area"].nunique()

77

In [992]:
df3.columns

Index(['year', 'ID', 'case_number', 'date', 'block', 'IUCR', 'primary_type',
       'description', 'location_description', 'crime_group1_prime',
       'crime_group3_iucr', 'drug_offence_type', 'arrest', 'domestic', 'Beat',
       'District', 'Ward', 'community_area', 'fbi_code', 'Latitude',
       'Longitude', 'Location', 'PRIMARY DESCRIPTION', 'SECONDARY DESCRIPTION',
       'INDEX CODE'],
      dtype='object')

In [993]:
# df3["primary_type"].unique()

In [994]:
# df3["arrest"].dtype

In [995]:
# 新しいデータフレームを作る

In [996]:
result = pd.crosstab(
    df3['community_area'],
    df3['primary_type']
).reindex(range(1, 78), fill_value=0).reset_index()


result.index.name = None
result.columns.name = None

# # 先頭行に年を追加
# result.insert(0, "year", year)



# #変数名のスペースを_に変更

# result.columns = [
#     f"{col.replace(' ', '_')}"
#     for col in result.columns
# ]



# result_drop = result.drop(columns = ["primary_type"]) 
# result.columns.name = None


In [997]:
# result

In [998]:
result.columns

Index(['community_area', 'arson', 'assault', 'battery', 'burglary',
       'concealed_carry_license_violation', 'criminal_damage',
       'criminal_sexual_assault', 'criminal_trespass', 'deceptive_practice',
       'gambling', 'homicide', 'human_trafficking',
       'interference_with_public_officer', 'intimidation', 'kidnapping',
       'liquor_law_violation', 'motor_vehicle_theft', 'narcotics', 'obscenity',
       'offense_involving_children', 'other_narcotic_violation',
       'other_offense', 'prostitution', 'public_indecency',
       'public_peace_violation', 'robbery', 'sex_offense', 'stalking', 'theft',
       'weapons_violation'],
      dtype='object')

In [999]:
result["community_area"].isna().sum()

np.int64(0)

#### 犯罪の分類別データを集計し，結合する

In [1000]:
group1 = pd.crosstab(
    df3['community_area'],
    df3['crime_group1_prime']
).reindex(range(1, 78), fill_value=0).reset_index()

group1.index.name = None
group1.columns.name = None
# result = result.merge(arrest_rate, on="community_area", how="left")
# result = result.merge(domestic_rate, on="community_area", how="left")

# group1

In [1001]:
group3 = pd.crosstab(
    df3['community_area'],
    df3['crime_group3_iucr']
).reindex(range(1, 78), fill_value=0).reset_index()

group3.index.name = None
group3.columns.name = None
# result = result.merge(arrest_rate, on="community_area", how="left")
# result = result.merge(domestic_rate, on="community_area", how="left")

# group3

In [1002]:
group5 = pd.crosstab(
    df3['community_area'],
    df3['drug_offence_type']
).reindex(range(1, 78), fill_value=0).reset_index()

group5.index.name = None
group5.columns.name = None
# result = result.merge(arrest_rate, on="community_area", how="left")
# result = result.merge(domestic_rate, on="community_area", how="left")

# group5

In [1003]:
merged.columns

Index(['year', 'community_area', 'other1', 'property1', 'public_order1',
       'violent1', 'child3', 'minor_offence3', 'minor_offences3', 'moral3',
       'narcotics3', 'others3', 'property3', 'sexual3', 'violent3', 'weapons3',
       'white_collar3', 'manufacture_deliver', 'possess', 'other1_arrest_rate',
       'property1_arrest_rate', 'public_order1_arrest_rate',
       'violent1_arrest_rate', 'child3_arrest_rate',
       'minor_offence3_arrest_rate', 'minor_offences3_arrest_rate',
       'moral3_arrest_rate', 'narcotics3_arrest_rate', 'others3_arrest_rate',
       'property3_arrest_rate', 'sexual3_arrest_rate', 'violent3_arrest_rate',
       'weapons3_arrest_rate', 'white_collar3_arrest_rate', 'arson', 'assault',
       'battery', 'burglary', 'concealed_carry_license_violation',
       'criminal_damage', 'criminal_sexual_assault', 'criminal_trespass',
       'deceptive_practice', 'gambling', 'homicide', 'human_trafficking',
       'interference_with_public_officer', 'intimidation'

#### Arrestの割合とDomesticの割合を計算する

In [1004]:
# df2["arrest"].dtype

In [1005]:
# arrest_rate = (
#     df2.groupby("community_area")["arrest"]
#        .mean()
#        .reset_index(name="arrest_rate")
# )

# domestic_rate = (
#     df2.groupby("community_area")["domestic"]
#        .mean()
#        .reset_index(name="domestic_rate")
# )
# result = result.merge(arrest_rate, on="community_area", how="left")
# result = result.merge(domestic_rate, on="community_area", how="left")

In [1006]:
# print(domestic_rate)

### コミュニティエリアごとの，各犯罪カテゴリーの逮捕率を表す表を作る。そして，結合する。

In [1007]:
group1_rate_table = pd.pivot_table(
    df3,
    index="community_area",
    columns='crime_group1_prime',
    values="arrest",
    aggfunc="mean"
)

group1_rate_table.columns = [
    f"{col.replace(' ', '_')}_arrest_rate"
    for col in group1_rate_table.columns
]


group1_rate_table = group1_rate_table.reset_index()

# # 先頭行に年を追加
# group1_rate_table.insert(0, "year", year)

In [1008]:
group3_rate_table = pd.pivot_table(
    df3,
    index="community_area",
    columns='crime_group3_iucr',
    values="arrest",
    aggfunc="mean"
)

group3_rate_table.columns = [
    f"{col.replace(' ', '_')}_arrest_rate"
    for col in group3_rate_table.columns
]


group3_rate_table = group3_rate_table.reset_index()

# # 先頭行に年を追加
# group3_rate_table.insert(0, "year", year)

In [1009]:
rate = group1_rate_table.merge(group3_rate_table, on="community_area", how="left")
# rate.insert(0, "year", year)

In [1010]:
# group1_rate_table
# group3_rate_table
rate

,community_area,other1_arrest_rate,property1_arrest_rate,public_order1_arrest_rate,violent1_arrest_rate,child3_arrest_rate,minor_offence3_arrest_rate,minor_offences3_arrest_rate,moral3_arrest_rate,narcotics3_arrest_rate,others3_arrest_rate,property3_arrest_rate,sexual3_arrest_rate,violent3_arrest_rate,weapons3_arrest_rate,white_collar3_arrest_rate
0,1.0,0.141079,0.117902,0.641577,0.209122,0.111111,0.666667,1.000000,1.000000,0.931818,0.413333,0.129787,0.104478,0.192536,0.634146,0.036290
1,2.0,0.119048,0.049061,0.445205,0.177521,0.000000,0.571429,0.750000,1.000000,0.875000,0.274854,0.057292,0.036364,0.166837,0.586207,0.000000
2,3.0,0.134715,0.100121,0.560000,0.139408,0.058824,0.142857,0.500000,1.000000,0.877551,0.337017,0.112073,0.053191,0.133707,0.800000,0.033133
3,4.0,0.062500,0.046167,0.540000,0.180294,0.000000,0.250000,0.000000,1.000000,1.000000,0.300000,0.051429,0.102564,0.170833,0.533333,0.016304
4,5.0,0.040000,0.021965,0.417910,0.083333,0.000000,0.600000,NaN,NaN,0.950000,0.090909,0.025815,0.055556,0.069231,0.600000,0.013986
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,73.0,0.287879,0.049007,0.680851,0.128951,0.000000,0.454545,0.714286,0.000000,0.888889,0.433673,0.050324,0.052632,0.121639,0.821782,0.071429
73,74.0,0.145455,0.056140,0.285714,0.090909,0.000000,1.000000,0.000000,NaN,NaN,0.285714,0.066964,0.100000,0.081522,0.666667,0.015152
74,75.0,0.162921,0.067202,0.646552,0.102253,0.000000,0.666667,0.333333,1.000000,0.958333,0.336634,0.073497,0.076923,0.086735,0.808511,0.054795
75,76.0,0.225225,0.036442,0.631922,0.190332,NaN,0.000000,0.000000,0.000000,0.960000,0.662162,0.037908,0.111111,0.169355,0.950495,0.027322


### 件数の表と逮捕率の表を結合する

In [1011]:
merged = group1.merge(group3, on="community_area", how="left")
merged.insert(0, "year", year)
merged = merged.merge(group5, on="community_area", how="left")
merged = merged.merge(rate, on="community_area", how="left")
merged = merged.merge(result, on="community_area", how="left")

In [1012]:
merged

,year,community_area,other1,property1,public_order1,violent1,child3,minor_offence3,minor_offences3,moral3,...,other_narcotic_violation,other_offense,prostitution,public_indecency,public_peace_violation,robbery,sex_offense,stalking,theft,weapons_violation
0,2025,1,241,2078,279,1162,18,9,3,2,...,0,241,0,1,7,90,30,12,1048,38
1,2025,2,252,1916,146,952,15,7,4,2,...,0,252,0,1,6,70,26,6,775,27
2,2025,3,193,2487,200,1284,17,7,4,1,...,0,192,1,0,9,76,41,9,1300,32
3,2025,4,112,1213,100,477,8,4,1,2,...,0,112,1,0,3,26,23,8,620,14
4,2025,5,75,865,67,240,5,5,0,0,...,0,75,0,0,3,17,7,7,347,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2025,73,264,1510,235,791,13,22,7,1,...,0,264,0,0,19,63,15,9,488,93
73,2025,74,55,285,21,165,2,1,1,0,...,0,55,0,0,2,7,4,3,105,3
74,2025,75,178,997,116,577,25,9,3,1,...,0,177,0,0,6,38,14,0,443,46
75,2025,76,111,933,307,331,0,54,23,2,...,0,111,0,0,77,2,6,3,537,33


### コミュニティエリアごとの，各犯罪のdomestic率を表す表を作る。そして，結合する。

In [1013]:
# domestic_rate_table = pd.pivot_table(
#     df2,
#     index="community_area",
#     columns="primary_type",
#     values="domestic",
#     aggfunc="mean"
# )

# domestic_rate_table.columns = [
#     f"{col.replace(' ', '_')}_domestic_rate"
#     for col in domestic_rate_table.columns
# ]


# domestic_rate_table = domestic_rate_table.reset_index()

# # 先頭行に年を追加
# domestic_rate_table.insert(0, "year", year)



# # # Community Areaという列名をcommunity_areaに変更
# # domestic_rate_table = domestic_rate_table.rename(columns={"Community Area": "community_area"})

In [1014]:
# domestic_rate_table

In [1015]:
# df2["primary_type"].value_counts()

### コミュニティエリアごとの犯罪発生件数をcsv出力

In [1016]:
merged.to_csv(rf"D:\JupyterLab Cleaned File\CommunityレベルCrime\Crime_community_area{year}.csv", index = False)

#### コミュニティエリアごとの逮捕率，domestic率をcsv出力

In [1017]:
# arrest_rate_table.to_csv(rf"D:\JupyterLab Cleaned File\CommunityレベルCrime\Arrest_Rate\Arrest_Rate{year}.csv", index = False)
# domestic_rate_table.to_csv(rf"D:\JupyterLab Cleaned File\CommunityレベルCrime\Domestic_Rate\Domestic_Rate{year}.csv", index = False)